# Cross-Framework Comparison - Logistic Regression

Initial environment configuration

In [1]:
import os
import sys

# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [2]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Global imports

In [3]:
from river import linear_model, optim
from tabulate import tabulate
import importlib.util
import sys
import os
import time
import tracemalloc

# We use an external library (scikit-learn) to compute the metrics consistently across models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

Dynamically load the custom LogisticRegression over the installed capymoa package

In [4]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

Global functions

In [5]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [6]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Accuracy", "F1", "Precision", "Recall"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    y_true = []
    y_pred = []

    # prequential evaluation
    for i, instance in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        # test
        votes = log_reg_capymoa.predict_proba(instance)
        pred = max(range(len(votes)), key=lambda j: votes[j])

        true_label = int(instance.y_index)

        y_true.append(true_label)
        y_pred.append(pred)

        # train
        log_reg_capymoa.train(instance)

    return _compute_sklearn_metrics(y_true, y_pred)

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    y_true = []
    y_pred = []

    # prequential evaluation
    for i, (x, y) in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        pred = log_reg_river.predict_one(x)

        if pred is None:
            pred = False

        y_true.append(y)
        y_pred.append(pred)

        log_reg_river.learn_one(x, y)

    return _compute_sklearn_metrics(y_true, y_pred)

def _compute_sklearn_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "F1": f1_score(y_true, y_pred, zero_division=0) * 100,
        "Precision": precision_score(y_true, y_pred, zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, zero_division=0) * 100,
    }

## Electricity dataset

In [7]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 69.93%    │ 69.93%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 75.02%    │ 75.02%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 71.86%    │ 71.86%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 78.48%    │ 78.48%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## ElectricityTiny dataset

In [8]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 62.85%    │ 62.85%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 38.85%    │ 38.85%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 55.79%    │ 55.79%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 29.80%    │ 29.80%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## RandomRBFGenerator

In [9]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 85.28%    │ 85.28%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 81.59%    │ 81.59%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 84.83%    │ 84.83%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 78.59%    │ 78.59%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## Hyper100k dataset

In [10]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 89.86%    │ 89.86%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 90.15%    │ 90.15%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 87.85%    │ 87.85%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 92.57%    │ 92.57%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## SEA dataset generator

In [11]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 82.91%    │ 82.91%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 87.05%    │ 87.05%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 84.69%    │ 84.69%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 89.56%    │ 89.56%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## HyperPlaneClassification dataset

In [12]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 89.87%    │ 89.87%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 90.18%    │ 90.18%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 87.71%    │ 87.71%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 92.78%    │ 92.78%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## RandomTreeGenerator

In [13]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 81.05%    │ 81.05%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 66.09%    │ 66.09%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 76.99%    │ 76.99%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 57.89%    │ 57.89%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


## Electricity dataset (changed model parameters)

### L2

In [14]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l2=0.01)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 68.49%    │ 68.49%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 74.33%    │ 74.33%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 69.97%    │ 69.97%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 79.27%    │ 79.27%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


### L1

In [15]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l1=0.01)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 65.76%    │ 65.76%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 73.34%    │ 73.34%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 66.43%    │ 66.43%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 81.86%    │ 81.86%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


### Learning rate

In [16]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 81.08%    │ 81.08%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 83.76%    │ 83.76%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 82.78%    │ 82.78%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 84.76%    │ 84.76%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


### Gradient clipping

In [17]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, clip=1)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 69.93%    │ 69.93%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 75.02%    │ 75.02%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 71.86%    │ 71.86%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 78.48%    │ 78.48%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛


### Bias initialization

In [18]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, bias_init=3)


--- Comparison CapyMOA vs River ---

╒═══════════╤═══════════╤═════════╤═════════╕
│ Metric    │ CapyMOA   │ River   │ Delta   │
╞═══════════╪═══════════╪═════════╪═════════╡
│ Accuracy  │ 70.62%    │ 70.62%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ F1        │ 75.65%    │ 75.65%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Precision │ 72.33%    │ 72.33%  │ +0.00%  │
├───────────┼───────────┼─────────┼─────────┤
│ Recall    │ 79.28%    │ 79.28%  │ +0.00%  │
╘═══════════╧═══════════╧═════════╧═════════╛
